# PA2 launcher 03: Task 4 safety generation, AI judging and the blind audit sheet
Settings: **Accelerator: GPU T4 x2**, **Internet: On**.
**Before running:** *Add Input → Your Work* → add the latest versions of launcher **01** and **02**, so the standard
DPO, PPO and GRPO adapters are restored. Afterwards: download `results/task4_safety/manual_audit_sheet.csv`,
label it by hand, then run `python -m task4_safety.evaluate_safety` (CPU, seconds) locally or in launcher 04.

In [ ]:
# ---- EDIT THIS: your public GitHub repository ----
REPO_URL = "https://github.com/<your-username>/<your-repo>.git"
BRANCH = "main"

import os, glob, shutil, subprocess, time, pathlib
WORK = "/tmp/pa2"             # big scratch disk: repo, course assets, model caches (not saved)
OUT = "/kaggle/working"       # saved with the notebook version: results/, outputs/, logs/
os.makedirs(f"{OUT}/logs", exist_ok=True)

def sh(cmd, cwd=None, log=None):
    """Run a shell command, stream its output into the notebook, stop the notebook on failure."""
    print(f"\n$ {cmd}", flush=True)
    p = subprocess.Popen("set -o pipefail; " + cmd, shell=True, cwd=cwd or WORK, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, executable="/bin/bash")
    with open(log or f"{OUT}/logs/main.log", "a") as f:
        for line in p.stdout:
            print(line, end="", flush=True)
            f.write(line)
    if p.wait():
        save()
        raise RuntimeError(f"command failed ({p.returncode}): {cmd}")

def save():
    """Copy results/ and outputs/ (small LoRA adapters) to the saved output folder."""
    for d in ("results", "outputs"):
        src = pathlib.Path(WORK, d)
        if src.exists():
            shutil.copytree(src, pathlib.Path(OUT, d), dirs_exist_ok=True)

def restore():
    """Merge results/ and outputs/ from any earlier notebook versions attached via 'Add Input'."""
    found = []
    for d in ("results", "outputs"):
        for src in glob.glob(f"/kaggle/input/**/{d}", recursive=True):
            if pathlib.Path(src).is_dir() and "/tmp/" not in src:
                shutil.copytree(src, pathlib.Path(WORK, d), dirs_exist_ok=True)
                found.append(src)
    print("restored from inputs:", found or "nothing attached")

def run_lanes(lanes, poll_s=120):
    """Run several command lists in parallel, one GPU each; print each lane's latest lines while waiting."""
    procs = {}
    for name, (gpu, cmds) in lanes.items():
        script = "set -euo pipefail\n" + "\n".join(cmds) + "\n"
        path = f"{OUT}/logs/{name}.sh"
        open(path, "w").write(script)
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), PYTHONUNBUFFERED="1")
        log = open(f"{OUT}/logs/{name}.log", "a")
        procs[name] = subprocess.Popen(["bash", path], cwd=WORK, env=env, stdout=log, stderr=subprocess.STDOUT)
        print(f"started lane {name} on GPU {gpu} -> logs/{name}.log")
    t0 = time.time()
    while any(p.poll() is None for p in procs.values()):
        time.sleep(poll_s)
        save()
        print(f"\n===== {(time.time() - t0) / 60:.0f} min =====")
        for name in procs:
            lines = open(f"{OUT}/logs/{name}.log").read().splitlines()
            state = "running" if procs[name].poll() is None else f"exit {procs[name].returncode}"
            print(f"[{name}: {state}]", *lines[-3:], sep="\n  ")
    save()
    bad = {n: p.returncode for n, p in procs.items() if p.returncode}
    if bad:
        raise RuntimeError(f"lanes failed: {bad}; see /kaggle/working/logs/<lane>.log")
    print("all lanes finished")

In [ ]:
sh(f"rm -rf {WORK} && git clone -q --branch {BRANCH} {REPO_URL} {WORK}", cwd="/tmp")
sh("git log --oneline -1")
sh("pip install -q -r requirements.txt")
sh("python -m scripts.download_assets && python -m scripts.validate_assets")
sh("python -m scripts.check_environment && python -m tests.test_objectives")
restore()

In [ ]:
for p in ("outputs/task1_dpo/standard", "outputs/task2_ppo/standard", "outputs/task3_grpo/standard"):
    assert os.path.exists(f"{WORK}/{p}/adapter_config.json"), f"missing {p}: attach launcher 01 and 02 outputs"

In [ ]:
run_lanes({
    'task4': (0, [
        'python -m task4_safety.generate_responses --config configs/feedback.yaml --batch-size 8',
        'python -m task4_safety.judge_responses --config configs/feedback.yaml',
        'python -m task4_safety.make_audit_sheet --config configs/feedback.yaml',
    ]),
})

In [ ]:
save()
sh("ls -R results | head -80")
print("Done. Download /kaggle/working/results (Output tab) and commit it to your repo.")